# 04 — Model Selection

Train and compare three models to predict next-gameweek FPL points:
1. **Linear Regression** — interpretable baseline
2. **Random Forest** — captures non-linear patterns
3. **XGBoost** — gradient boosting, typically best for tabular data

**Train/test split:** Time-based. Train on GW4-22, test on GW23-29.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

plt.style.use('ggplot')
pd.set_option('display.max_columns', 40)

## 1. Load feature matrix and split

In [ ]:
df = pd.read_csv(project_root / 'data' / 'processed' / 'features.csv')

id_cols = ['player_id', 'web_name', 'position', 'team_name', 'round']
target_col = 'target'
feature_cols = [c for c in df.columns if c not in id_cols + [target_col]]

# Time-based split
TRAIN_END_GW = 22
train = df[df['round'] <= TRAIN_END_GW]
test = df[df['round'] > TRAIN_END_GW]

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

print(f'Features: {len(feature_cols)}')
print(f'Train: {len(train):,} rows (GW4-{TRAIN_END_GW})')
print(f'Test:  {len(test):,} rows (GW{TRAIN_END_GW+1}-{df["round"].max()})')
print(f'\nTarget mean — train: {y_train.mean():.2f}, test: {y_test.mean():.2f}')

## 2. Train all three models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=20,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1
    ),
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test)
    
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': root_mean_squared_error(y_test, y_pred),
        'R²': r2_score(y_test, y_pred),
    }
    print(f'  MAE={results[name]["MAE"]:.3f}  RMSE={results[name]["RMSE"]:.3f}  R²={results[name]["R²"]:.3f}')

print('\nDone!')

## 3. Model comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.round(3)

print('=== Model Comparison ===')
print(results_df.to_string())

best_name = results_df['MAE'].idxmin()
print(f'\nBest model by MAE: {best_name}')

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['MAE', 'RMSE', 'R²']):
    results_df[metric].plot(kind='bar', ax=axes[i], color=['steelblue', 'forestgreen', 'coral'])
    axes[i].set_title(metric)
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Feature importance (best model)

In [ ]:
best_model = trained_models[best_name]

# Get feature importances
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_)

feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
feat_imp.tail(20).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Top 20 Feature Importances — {best_name}')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
for feat, imp in feat_imp.tail(10).iloc[::-1].items():
    print(f'  {feat:30s}  {imp:.4f}')

## 5. Linear Regression coefficients (the equation)

Even if Linear Regression isn't the best model, its coefficients give us an interpretable formula.

In [ ]:
lr = trained_models['Linear Regression']
coefs = pd.Series(lr.coef_, index=feature_cols).sort_values(ascending=False)

print(f'Intercept: {lr.intercept_:.3f}')
print(f'\nTop 10 positive coefficients (increase points):')
for feat, coef in coefs.head(10).items():
    print(f'  {feat:30s}  {coef:+.4f}')

print(f'\nTop 10 negative coefficients (decrease points):')
for feat, coef in coefs.tail(10).items():
    print(f'  {feat:30s}  {coef:+.4f}')

## 6. Residual analysis — where does the model fail?

In [ ]:
y_pred_best = best_model.predict(X_test)
test_analysis = test[id_cols].copy()
test_analysis['actual'] = y_test.values
test_analysis['predicted'] = y_pred_best
test_analysis['error'] = test_analysis['actual'] - test_analysis['predicted']
test_analysis['abs_error'] = test_analysis['error'].abs()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residual distribution
axes[0].hist(test_analysis['error'], bins=40, edgecolor='black', alpha=0.7)
axes[0].set_title('Prediction Error Distribution')
axes[0].set_xlabel('Actual - Predicted')
axes[0].axvline(x=0, color='red', linestyle='--')

# Predicted vs Actual
axes[1].scatter(test_analysis['predicted'], test_analysis['actual'], alpha=0.1, s=5)
max_val = max(test_analysis['actual'].max(), test_analysis['predicted'].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', alpha=0.5)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Predicted vs Actual')

# MAE by position
pos_mae = test_analysis.groupby('position')['abs_error'].mean().sort_values()
pos_mae.plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('MAE by Position')
axes[2].set_ylabel('MAE')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('MAE by position:')
print(pos_mae.round(3).to_string())

In [ ]:
# Biggest misses — where the model got it most wrong
print('=== Biggest underpredictions (player scored way more than predicted) ===')
print(test_analysis.nlargest(10, 'error')[['web_name', 'position', 'round', 'actual', 'predicted', 'error']].to_string(index=False))

print('\n=== Biggest overpredictions (player scored way less than predicted) ===')
print(test_analysis.nsmallest(10, 'error')[['web_name', 'position', 'round', 'actual', 'predicted', 'error']].to_string(index=False))

## 7. Gameweek-level accuracy

How well does the model rank players within each gameweek?

In [ ]:
# Per-gameweek MAE
gw_metrics = test_analysis.groupby('round').agg(
    mae=('abs_error', 'mean'),
    avg_actual=('actual', 'mean'),
    count=('actual', 'count')
).round(3)

print('=== Per-Gameweek MAE ===')
print(gw_metrics.to_string())

# How often does the model's top 15 actually appear in the real top 30?
print('\n=== Top-player prediction accuracy ===')
for gw in sorted(test_analysis['round'].unique()):
    gw_data = test_analysis[test_analysis['round'] == gw]
    pred_top15 = set(gw_data.nlargest(15, 'predicted')['player_id'])
    actual_top30 = set(gw_data.nlargest(30, 'actual')['player_id'])
    overlap = len(pred_top15 & actual_top30)
    print(f'  GW{gw}: {overlap}/15 predicted top players appeared in actual top 30')

## 8. Save best model

In [ ]:
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Save model
model_path = models_dir / 'best_model.joblib'
joblib.dump(best_model, model_path)

# Save feature columns and metadata
metadata = {
    'model_name': best_name,
    'feature_columns': feature_cols,
    'train_gw_range': [4, TRAIN_END_GW],
    'test_gw_range': [TRAIN_END_GW + 1, int(df['round'].max())],
    'metrics': results[best_name],
}
with open(models_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Also save all model results for comparison
with open(models_dir / 'all_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved {best_name} to {model_path}')
print(f'Model size: {model_path.stat().st_size / 1024:.0f} KB')
print(f'Metadata saved to {models_dir / "model_metadata.json"}')